In [1]:
from cbr_inflation import get_inflation, get_latest_inflation, get_latest_target
from datetime import date
import pandas as pd
inf = get_inflation()
inf_d = inf.copy()
inf_d['Год'] = inf_d['date'].dt.year
inf_d.rename(columns={
    'inflation': 'Инфляция',
    'target': 'Цель по инфляции'},inplace=True)
inf_d = inf_d[['Год','Инфляция']]
inf_d = inf_d.tail(1)
inf_d

,Год,Инфляция
113,2026,6.02


In [2]:
current_year = date.today().year
forecast_years = [ current_year + 1, current_year + 2]

In [3]:
inf2 = pd.DataFrame({
    'Год': forecast_years,
    'Инфляция': inf['target'].iloc[-1]})
inf2

,Год,Инфляция
0,2027,4.0
1,2028,4.0


In [4]:
inf_res = pd.concat([inf_d,inf2],ignore_index=True)

In [5]:
import requests
import pandas as pd

BASE_URL = "http://www.cbr.ru/dataservice"

# Параметры для депозитов физических лиц
PUBLICATION_ID = 18      # В целом по РФ (депозиты)
DATASET_ID = 37
# Ставки по вкладам физических лиц

# Запрашиваем данные
params = {
    "publicationId": PUBLICATION_ID,
    "y1": 2020,
    "y2": 2026,
    "i_ids": [DATASET_ID],
    "m1_ids": [2],
    "m2_ids": [7]
}

response = requests.get(f"{BASE_URL}/dataEx", params=params)
data = response.json()
raw = data.get("RawData", [])

# Преобразуем в DataFrame
df = pd.DataFrame(raw)
# Переименуем колонки для удобства
df.rename(columns={
    'period': 'period_name',
    'date': 'date_str',
    'value': 'rate',
    'measure_1_id': 'currency_id',
    'measure_2_id': 'term_id',
    'period_id': 'period_id',
    'rowId': 'row_id'
}, inplace=True)
# Преобразуем дату в datetime
df['date'] = pd.to_datetime(df['date_str'], format='%d.%m.%Y')
df = df.sort_values('date').reset_index(drop=True)
# Добавим понятные названия для валют и сроков (можно будет подтянуть из /measures)
# Но для простоты оставим как есть

df = df[['date', 'rate', 'currency_id', 'term_id', 'period_name']]
sd = df.tail(1)
value = sd['rate'].iloc[0]

In [10]:
DEPOSIT_DECREMENT = 2.5
base = value
inf_res['Ставка депозита'] = base - (DEPOSIT_DECREMENT/100) * inf_res.index
inf_res[['Инфляция','Ставка депозита']] = inf_res[['Инфляция','Ставка депозита']]/100
inf_res

,Год,Инфляция,Ставка депозита
0,2026,0.0602,0.12840
1,2027,0.0400,0.12815
2,2028,0.0400,0.12790


In [7]:
6.02/100

0.0602

In [8]:
from cbr_inflation import get_inflation
import requests
import pandas as pd
from datetime import datetime

# 1. Получаем инфляцию
inf = get_inflation()
inf_d = inf.copy()
inf_d['Год'] = inf_d['date'].dt.year
inf_d.rename(columns={'inflation': 'Инфляция'}, inplace=True)
inf_d = inf_d[['Год', 'Инфляция']]
inf_d = inf_d.tail(1)  # последний доступный год

# 2. Экстраполируем на будущие годы (используем целевую инфляцию)
current_year = datetime.today().year
forecast_years = [current_year + 1, current_year + 2]
forecast_inflation = inf['target'].iloc[-1]  # последняя цель по инфляции
inf_forecast = pd.DataFrame({
    'Год': forecast_years,
    'Инфляция': [forecast_inflation] * len(forecast_years)
})
inf_res = pd.concat([inf_d, inf_forecast], ignore_index=True)

# 3. Получаем депозитную ставку (REST API)
BASE_URL = "http://www.cbr.ru/dataservice"
params = {
    "publicationId": 18,
    "y1": 2020,
    "y2": 2026,
    "i_ids": [37],
    "m1_ids": [2],
    "m2_ids": [7]
}
response = requests.get(f"{BASE_URL}/dataEx", params=params)
data = response.json()
raw = data.get("RawData", [])
if not raw:
    raise ValueError("Нет данных по депозитам")
df_dep = pd.DataFrame(raw)
df_dep['date'] = pd.to_datetime(df_dep['date'], format='%d.%m.%Y')
df_dep = df_dep.sort_values('date')
latest_rate = df_dep['value'].iloc[-1]

# 4. Рассчитываем ставку депозита с линейным уменьшением
DEPOSIT_DECREMENT = 2.5
base_rate = latest_rate
inf_res['Ставка депозита'] = base_rate - DEPOSIT_DECREMENT * range(len(inf_res))

print(inf_res)

TypeError: unsupported operand type(s) for *: 'float' and 'range'